In [18]:
# Resolve project paths when launched from the root or a project subfolder.
from pathlib import Path

PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "trial_manifest.json").is_file() and (p / "analysis").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch this notebook from the new G&P project folder or analysis folder.")
DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


Imports + dataset location

In [19]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_ROOT = PROJECT_ROOT / "data"

PEOPLE = [
    "Buddhini",
    "Harshika",
    "Muthuni",
    "Nadeera",
    "Ruwani",
    "Thenuri"
]

print("Dataset root:", DATA_ROOT)
print("\nPeople:")

for person in PEOPLE:
    path = DATA_ROOT / person
    print(f"{person:10s} -> {'FOUND' if path.exists() else 'NOT FOUND'}")

Dataset root: C:\Users\MSI\OneDrive\Desktop\Gait and Posture\new G&P\data

People:
Buddhini   -> FOUND
Harshika   -> FOUND
Muthuni    -> FOUND
Nadeera    -> FOUND
Ruwani     -> FOUND
Thenuri    -> FOUND


Discover the actual dataset structure

In [20]:
records = []

for person in PEOPLE:
    person_path = DATA_ROOT / person

    for file in person_path.rglob("*.csv"):
        records.append({
            "person": person,
            "file_name": file.name,
            "relative_path": str(file.relative_to(person_path)),
            "full_path": str(file)
        })

files_df = pd.DataFrame(records)

# Summary
summary = (
    files_df
    .groupby("person")
    .agg(
        total_csv_files=("file_name", "size"),
        unique_file_types=("file_name", "nunique")
    )
    .reset_index()
)

display(summary)

print("\nTotal CSV files:", len(files_df))

,person,total_csv_files,unique_file_types
0,Buddhini,75,8
1,Harshika,45,8
2,Muthuni,21,7
3,Nadeera,42,7
4,Ruwani,8,8
5,Thenuri,56,7



Total CSV files: 247


Audit the limb_angle_mode.csv files

In [21]:
# Keep only limb_angle_mode.csv files
angle_files = files_df[
    files_df["file_name"] == "limb_angle_mode.csv"
].copy()

# Extract trial and phase information from the path
def extract_structure(row):
    path = Path(row["relative_path"])
    parts = path.parts

    trial = parts[0] if len(parts) >= 1 else None

    phase_folder = parts[1] if len(parts) >= 2 else None

    if phase_folder and "Phase_" in phase_folder:
        phase = phase_folder.split("Phase_")[-1]
    else:
        phase = None

    return pd.Series({
        "trial": trial,
        "phase": phase
    })

angle_files[["trial", "phase"]] = angle_files.apply(
    extract_structure,
    axis=1
)

# Sort nicely
angle_files["trial_num"] = pd.to_numeric(
    angle_files["trial"],
    errors="coerce"
)

angle_files = angle_files.sort_values(
    ["person", "trial_num", "phase"]
)

# Summary by person
angle_summary = (
    angle_files
    .groupby("person")
    .agg(
        angle_files=("file_name", "size"),
        trials=("trial", "nunique"),
        phases=("phase", "nunique")
    )
    .reset_index()
)

display(angle_summary)

print("\nTotal limb_angle_mode.csv files:", len(angle_files))

,person,angle_files,trials,phases
0,Buddhini,10,9,2
1,Harshika,6,5,2
2,Muthuni,3,3,2
3,Nadeera,6,4,2
4,Ruwani,1,1,1
5,Thenuri,8,5,2



Total limb_angle_mode.csv files: 34


4. Check which trial has which phase

In [22]:
# CELL 4 — Show trials and phases clearly

trial_phase_table = (
    angle_files
    .groupby(["person", "trial"], dropna=False)
    .agg(
        phases=("phase", lambda x: ", ".join(
            sorted(str(v) for v in x.dropna())
        )),
        angle_files=("file_name", "size")
    )
    .reset_index()
)

# Sort trials numerically
trial_phase_table["trial_num"] = pd.to_numeric(
    trial_phase_table["trial"],
    errors="coerce"
)

trial_phase_table = (
    trial_phase_table
    .sort_values(["person", "trial_num"])
    .drop(columns="trial_num")
    .reset_index(drop=True)
)

display(trial_phase_table)

,person,trial,phases,angle_files
0,Buddhini,1,02,1
1,Buddhini,2,02,1
2,Buddhini,3,02,1
3,Buddhini,4,02,1
4,Buddhini,5,02,1
5,Buddhini,6,02,1
6,Buddhini,7,"01, 02",2
7,Buddhini,8,02,1
8,Buddhini,9,02,1
9,Harshika,1,02,1


5. Any phase names failed to extract

In [23]:
# CELL 5 — Check whether any phase names failed to extract

missing_phase = angle_files[
    angle_files["phase"].isna()
][["person", "trial", "relative_path"]]

print("Number of files with unknown phase:", len(missing_phase))

if len(missing_phase) > 0:
    display(missing_phase)
else:
    print("Good: phase was identified for every angle file.")

Number of files with unknown phase: 1


,person,trial,relative_path
191,Thenuri,5,5\combined_log_wide_1773380416\limb_angle_mode.csv


Inspect the structure inside all 34 angle files

In [24]:
# CELL 6 — Audit contents of every limb_angle_mode.csv

audit_records = []

for _, row in angle_files.iterrows():
    file_path = Path(row["full_path"])

    try:
        df = pd.read_csv(file_path)

        original_angle_cols = [
            col for col in df.columns
            if col.endswith("_original_angle")
        ]

        audit_records.append({
            "person": row["person"],
            "trial": row["trial"],
            "phase": row["phase"] if pd.notna(row["phase"]) else "UNKNOWN",
            "rows": len(df),
            "columns": len(df.columns),
            "original_angle_cols": len(original_angle_cols),
            "missing_values": int(df.isna().sum().sum()),
            "has_time": "time" in df.columns,
            "has_segment_label": "segment_label" in df.columns,
            "has_cycle_index": "cycle_index" in df.columns
        })

    except Exception as e:
        audit_records.append({
            "person": row["person"],
            "trial": row["trial"],
            "phase": row["phase"] if pd.notna(row["phase"]) else "UNKNOWN",
            "rows": None,
            "columns": None,
            "original_angle_cols": None,
            "missing_values": None,
            "has_time": None,
            "has_segment_label": None,
            "has_cycle_index": None,
            "error": str(e)
        })

audit_df = pd.DataFrame(audit_records)

display(audit_df)

,person,trial,phase,rows,columns,original_angle_cols,missing_values,has_time,has_segment_label,has_cycle_index
0,Buddhini,1,02,2087,76,18,0,True,True,True
1,Buddhini,2,02,2252,76,18,0,True,True,True
2,Buddhini,3,02,2592,76,18,0,True,True,True
3,Buddhini,4,02,2455,76,18,0,True,True,True
4,Buddhini,5,02,3443,76,18,0,True,True,True
5,Buddhini,6,02,3264,76,18,0,True,True,True
6,Buddhini,7,01,3461,76,18,0,True,True,True
7,Buddhini,7,02,3854,76,18,0,True,True,True
8,Buddhini,8,02,4173,76,18,0,True,True,True
9,Buddhini,9,02,2360,76,18,0,True,True,True


Check whether all recordings have the same schema:

In [25]:
# CELL 7 — Overall consistency summary

print("Rows per file:")
print(audit_df["rows"].describe())

print("\nDifferent numbers of columns:")
print(audit_df["columns"].value_counts().sort_index())

print("\nOriginal angle column counts:")
print(audit_df["original_angle_cols"].value_counts().sort_index())

print("\nFiles with missing values:")
display(
    audit_df[audit_df["missing_values"] > 0]
)

print("\nFiles missing required metadata columns:")
display(
    audit_df[
        (~audit_df["has_time"]) |
        (~audit_df["has_segment_label"]) |
        (~audit_df["has_cycle_index"])
    ]
)

Rows per file:
count      34.000000
mean     2993.382353
std       854.024023
min      1654.000000
25%      2214.500000
50%      2882.500000
75%      3742.750000
max      4597.000000
Name: rows, dtype: float64

Different numbers of columns:
columns
76    34
Name: count, dtype: int64

Original angle column counts:
original_angle_cols
18    34
Name: count, dtype: int64

Files with missing values:


,person,trial,phase,rows,columns,original_angle_cols,missing_values,has_time,has_segment_label,has_cycle_index



Files missing required metadata columns:


,person,trial,phase,rows,columns,original_angle_cols,missing_values,has_time,has_segment_label,has_cycle_index


Inspect columns

In [26]:
# CELL 8 — Inspect column names from a representative file

sample_path = Path(angle_files.iloc[0]["full_path"])
sample_df = pd.read_csv(sample_path)

print("Sample file:")
print(sample_path)

print("\nShape:", sample_df.shape)

print("\nAll columns:")
for i, col in enumerate(sample_df.columns, start=1):
    print(f"{i:2d}. {col}")

Sample file:
C:\Users\MSI\OneDrive\Desktop\Gait and Posture\new G&P\data\Buddhini\1\combined_log_wide_1757660547_Phase_02\limb_angle_mode.csv

Shape: (2087, 76)

All columns:
 1. frame
 2. time
 3. left_ankle_to_foot_mode_1
 4. left_ankle_to_foot_mode_2
 5. left_ankle_to_foot_mode_3
 6. left_elbow_to_wrist_mode_1
 7. left_elbow_to_wrist_mode_2
 8. left_elbow_to_wrist_mode_3
 9. left_hip_to_knee_mode_1
10. left_hip_to_knee_mode_2
11. left_hip_to_knee_mode_3
12. left_knee_to_ankle_mode_1
13. left_knee_to_ankle_mode_2
14. left_knee_to_ankle_mode_3
15. left_shoulder_to_elbow_mode_1
16. left_shoulder_to_elbow_mode_2
17. left_shoulder_to_elbow_mode_3
18. left_wrist_to_hand_mode_1
19. left_wrist_to_hand_mode_2
20. left_wrist_to_hand_mode_3
21. mid_to_upper_spine_mode_1
22. mid_to_upper_spine_mode_2
23. mid_to_upper_spine_mode_3
24. neck_to_left_shoulder_mode_1
25. neck_to_left_shoulder_mode_2
26. neck_to_left_shoulder_mode_3
27. neck_to_right_shoulder_mode_1
28. neck_to_right_shoulder_mode_2


In [27]:
# CELL 9 — Identify the original-angle signals

original_angle_cols = [
    col for col in sample_df.columns
    if col.endswith("_original_angle")
]

print(f"Original angle signals: {len(original_angle_cols)}\n")

for i, col in enumerate(original_angle_cols, start=1):
    print(f"{i:2d}. {col}")

Original angle signals: 18

 1. left_ankle_to_foot_original_angle
 2. left_elbow_to_wrist_original_angle
 3. left_hip_to_knee_original_angle
 4. left_knee_to_ankle_original_angle
 5. left_shoulder_to_elbow_original_angle
 6. left_wrist_to_hand_original_angle
 7. mid_to_upper_spine_original_angle
 8. neck_to_left_shoulder_original_angle
 9. neck_to_right_shoulder_original_angle
10. pelvis_to_left_hip_original_angle
11. pelvis_to_lower_spine_original_angle
12. pelvis_to_right_hip_original_angle
13. right_ankle_to_foot_original_angle
14. right_elbow_to_wrist_original_angle
15. right_hip_to_knee_original_angle
16. right_knee_to_ankle_original_angle
17. right_shoulder_to_elbow_original_angle
18. right_wrist_to_hand_original_angle


In [28]:
# CELL 10 — Inspect segment labels and cycle indices across all files

cycle_audit = []

for _, row in angle_files.iterrows():

    df = pd.read_csv(row["full_path"])

    # Periodic walking portion
    periodic = df[df["segment_label"] == "periodic"].copy()

    # Valid annotated cycles
    valid_cycles = periodic[
        periodic["cycle_index"] >= 0
    ]

    cycle_audit.append({
        "person": row["person"],
        "trial": row["trial"],
        "phase": (
            row["phase"]
            if pd.notna(row["phase"])
            else "UNKNOWN"
        ),

        "total_rows": len(df),

        "periodic_rows": len(periodic),

        "periodic_percent": (
            len(periodic) / len(df) * 100
            if len(df) > 0 else 0
        ),

        "valid_cycles": valid_cycles["cycle_index"].nunique(),

        "cycle_min": (
            valid_cycles["cycle_index"].min()
            if len(valid_cycles) else None
        ),

        "cycle_max": (
            valid_cycles["cycle_index"].max()
            if len(valid_cycles) else None
        )
    })

cycle_audit_df = pd.DataFrame(cycle_audit)

display(cycle_audit_df)

print(
    "\nTotal detected cycles:",
    cycle_audit_df["valid_cycles"].sum()
)

,person,trial,phase,total_rows,periodic_rows,periodic_percent,valid_cycles,cycle_min,cycle_max
0,Buddhini,1,02,2087,1200,57.498802,10,1.0,10.0
1,Buddhini,2,02,2252,1320,58.614565,15,1.0,15.0
2,Buddhini,3,02,2592,1110,42.824074,9,1.0,9.0
3,Buddhini,4,02,2455,1080,43.991853,9,1.0,9.0
4,Buddhini,5,02,3443,0,0.000000,0,NaN,NaN
5,Buddhini,6,02,3264,0,0.000000,0,NaN,NaN
6,Buddhini,7,01,3461,1980,57.208899,16,1.0,16.0
7,Buddhini,7,02,3854,0,0.000000,0,NaN,NaN
8,Buddhini,8,02,4173,0,0.000000,0,NaN,NaN
9,Buddhini,9,02,2360,1830,77.542373,18,1.0,18.0



Total detected cycles: 446


In [29]:
# CELL 11 — Check segment labels used in every file

label_records = []

for _, row in angle_files.iterrows():

    df = pd.read_csv(row["full_path"])

    counts = df["segment_label"].value_counts(
        dropna=False
    ).to_dict()

    label_records.append({
        "person": row["person"],
        "trial": row["trial"],
        "phase": (
            row["phase"]
            if pd.notna(row["phase"])
            else "UNKNOWN"
        ),
        "labels": counts
    })

label_df = pd.DataFrame(label_records)

pd.set_option("display.max_colwidth", None)

display(label_df)

,person,trial,phase,labels
0,Buddhini,1,02,"{'periodic': 1200, 'end_transient': 527, 'start_transient': 360}"
1,Buddhini,2,02,"{'periodic': 1320, 'end_transient': 662, 'start_transient': 270}"
2,Buddhini,3,02,"{'end_transient': 1152, 'periodic': 1110, 'start_transient': 330}"
3,Buddhini,4,02,"{'periodic': 1080, 'start_transient': 810, 'end_transient': 565}"
4,Buddhini,5,02,{'start_transient': 3443}
5,Buddhini,6,02,{'start_transient': 3264}
6,Buddhini,7,01,"{'periodic': 1980, 'start_transient': 960, 'end_transient': 521}"
7,Buddhini,7,02,{'start_transient': 3854}
8,Buddhini,8,02,{'start_transient': 4173}
9,Buddhini,9,02,"{'periodic': 1830, 'start_transient': 270, 'end_transient': 260}"


In [30]:
# CELL 12 — Investigate recordings with no periodic cycles

zero_cycle_files = cycle_audit_df[
    cycle_audit_df["valid_cycles"] == 0
][["person", "trial", "phase"]]

display(zero_cycle_files)

print("\nSegment labels in zero-cycle recordings:\n")

for _, z in zero_cycle_files.iterrows():

    match = angle_files[
        (angle_files["person"] == z["person"]) &
        (angle_files["trial"].astype(str) == str(z["trial"]))
    ]

    # Match phase too
    if z["phase"] != "UNKNOWN":
        match = match[
            match["phase"].astype(str) == str(z["phase"])
        ]

    if len(match) == 0:
        continue

    df = pd.read_csv(match.iloc[0]["full_path"])

    print(
        f"{z['person']} | Trial {z['trial']} | Phase {z['phase']}"
    )

    print("segment_label:")
    print(df["segment_label"].value_counts(dropna=False))

    print("\ncycle_index:")
    print(df["cycle_index"].value_counts(dropna=False).head(15))

    print("-" * 60)

,person,trial,phase
4,Buddhini,5,02
5,Buddhini,6,02
7,Buddhini,7,02
8,Buddhini,8,02
33,Thenuri,5,UNKNOWN



Segment labels in zero-cycle recordings:

Buddhini | Trial 5 | Phase 02
segment_label:
segment_label
start_transient    3443
Name: count, dtype: int64

cycle_index:
cycle_index
-1    3443
Name: count, dtype: int64
------------------------------------------------------------
Buddhini | Trial 6 | Phase 02
segment_label:
segment_label
start_transient    3264
Name: count, dtype: int64

cycle_index:
cycle_index
-1    3264
Name: count, dtype: int64
------------------------------------------------------------
Buddhini | Trial 7 | Phase 02
segment_label:
segment_label
start_transient    3854
Name: count, dtype: int64

cycle_index:
cycle_index
-1    3854
Name: count, dtype: int64
------------------------------------------------------------
Buddhini | Trial 8 | Phase 02
segment_label:
segment_label
start_transient    4173
Name: count, dtype: int64

cycle_index:
cycle_index
-1    4173
Name: count, dtype: int64
------------------------------------------------------------
Thenuri | Trial 5 | Phase

In [31]:
# CELL 13 — Check whether angle recordings are exact duplicates

import hashlib

def signal_hash(file_path):
    df = pd.read_csv(file_path)

    # Hash only the 18 original-angle signals
    signal_data = df[original_angle_cols].to_numpy()

    return hashlib.sha256(
        signal_data.tobytes()
    ).hexdigest()


duplicate_records = []

for _, row in angle_files.iterrows():

    duplicate_records.append({
        "person": row["person"],
        "trial": row["trial"],
        "phase": (
            row["phase"]
            if pd.notna(row["phase"])
            else "UNKNOWN"
        ),
        "rows": pd.read_csv(row["full_path"], usecols=["frame"]).shape[0],
        "hash": signal_hash(row["full_path"])
    })

duplicate_df = pd.DataFrame(duplicate_records)

duplicates = duplicate_df[
    duplicate_df.duplicated("hash", keep=False)
].sort_values("hash")

if len(duplicates) == 0:
    print("GOOD: No exact duplicate original-angle recordings found.")
else:
    print("WARNING: Exact duplicate recordings found:")
    display(
        duplicates[
            ["person", "trial", "phase", "rows", "hash"]
        ]
    )

GOOD: No exact duplicate original-angle recordings found.


In [32]:
# CELL 14 — Summarize usable periodic recordings

usable = cycle_audit_df[
    cycle_audit_df["valid_cycles"] > 0
].copy()

usable_summary = (
    usable
    .groupby("person")
    .agg(
        usable_files=("valid_cycles", "size"),
        usable_trials=("trial", "nunique"),
        total_cycles=("valid_cycles", "sum"),
        mean_cycles_per_file=("valid_cycles", "mean")
    )
    .reset_index()
)

usable_summary["mean_cycles_per_file"] = (
    usable_summary["mean_cycles_per_file"].round(1)
)

display(usable_summary)

print(
    "\nTotal usable files:",
    len(usable)
)

print(
    "Total usable person-trial combinations:",
    usable[["person", "trial"]]
    .drop_duplicates()
    .shape[0]
)

print(
    "Total valid gait cycles:",
    usable["valid_cycles"].sum()
)

,person,usable_files,usable_trials,total_cycles,mean_cycles_per_file
0,Buddhini,6,6,77,12.8
1,Harshika,6,5,59,9.8
2,Muthuni,3,3,30,10.0
3,Nadeera,6,4,124,20.7
4,Ruwani,1,1,11,11.0
5,Thenuri,7,4,145,20.7



Total usable files: 29
Total usable person-trial combinations: 23
Total valid gait cycles: 446
